# Vidéos satisfying en 3D (Blender) — rendu par lots sur GPU

1. *Settings* : **Accelerator GPU** (T4 ou P100), **Internet on**. Le jeton GitHub doit être dans *Add-ons → Secrets* sous le nom `GITHUB_TOKEN` (comme pour l'autre notebook).
2. **Save Version → Save & Run All** : le notebook tourne seul jusqu'à 12 h en arrière-plan et produit autant de vidéos que possible.
3. À la fin, les vidéos (et leur légende) sont dans l'onglet *Output* : `satisfying3d.zip`.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

BRANCHE = 'claude/video-4k-high-fps-b7phx9'
CODE = '/kaggle/working/code'
try:
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    raise SystemExit("Secret GITHUB_TOKEN non attaché à CE notebook : menu Add-ons → Secrets, "
                     "cochez GITHUB_TOKEN, puis relancez cette cellule.")
if not os.path.exists(CODE):
    r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCHE,
                        f'https://{token}@github.com/Amola07/Mon_Dashboard_Client.git', CODE],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***') or 'Code récupéré')
os.chdir(CODE)

In [ ]:
!pip install -q skia-python
!bash scripts/setup_tools.sh
os.environ['PATH'] = f"{CODE}/tools/ffmpeg/bin:" + os.environ['PATH']
!nvidia-smi --query-gpu=name,driver_version --format=csv

## Test rapide (une image)
Vérifie que Blender voit bien le GPU (ligne « Rendu sur GPU … ») et donne le temps par image.

In [ ]:
!python -m satisfying.render3d --still 40 --scale 0.5 --out /kaggle/working/satisfying3d

## Rendu par lots
`HEURES` : durée maximale ; le notebook s'arrête avant la limite de 12 h de Kaggle. Chaque vidéo tire au hasard sa palette,
son timbre et sa trajectoire. `SCALE = 0.5` rend en 540×960 puis agrandit en 1080×1920 ; `SAMPLES` règle la finesse de
la lumière (plus haut = plus beau, plus lent).

In [ ]:
import time, glob, shutil
HEURES = 10.5
SCALE = 0.5
SAMPLES = 64
debut = time.time()
estimation = None
while True:
    ecoule = (time.time() - debut) / 3600
    if estimation and ecoule + estimation > HEURES:
        break
    t = time.time()
    r = subprocess.run(['python', '-m', 'satisfying.render3d', '--scale', str(SCALE), '--samples', str(SAMPLES),
                        '--out', '/kaggle/working/satisfying3d'])
    if r.returncode:
        print('Échec de ce rendu, on passe au suivant')
    estimation = (time.time() - t) / 3600
    print(f'--- {len(glob.glob("/kaggle/working/satisfying3d/*.mp4"))} vidéo(s), {estimation * 60:.0f} min pour la dernière')
shutil.make_archive('/kaggle/working/satisfying3d', 'zip', '/kaggle/working/satisfying3d')
print('Terminé : satisfying3d.zip dans l\'onglet Output')